In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# os.environ["JAX_PLATFORM_NAME"] = "cpu"

In [3]:
import h5py, os, tqdm, glob, scipy
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import jax
import jax.numpy as jnp
import jax_cosmo as jc

import optax
from flax import nnx
import orbax.checkpoint as ocp
import jraph

import diffrax
from diffrax import diffeqsolve, ODETerm, Dopri5, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize
from jax.experimental.ode import odeint

import jaxpm
from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.pm import linear_field, lpt, make_ode_fn, pm_forces, make_ode_fn_diffrax, make_ode_fn
from jaxpm.kernels import fftk, gradient_kernel, invlaplace_kernel, longrange_kernel, invnabla_kernel
from jaxpm.utils import power_spectrum, cross_correlation_coefficients
from jaxpm.nn import MLP, ResNet3D, ResNetBlock3D, GraphConvolution, CNN, HybridNet, AttentionGNN
from jaxpm import camels, plotting, hpm, nn
from jaxpm.graph import jax_get_knn

# print(jax.devices("gpu"))
print(jax.default_backend())

gpu


# configuration

In [4]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim
mesh_shape = [mesh_per_dim] * 3
box_size = [float(mesh_per_dim)] * 3

# CAMELS

In [5]:
# SIM = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0"
# SIM = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1"
SIM = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2"

out_dict = camels.load_CV_snapshots(
    SIM,
    mesh_per_dim,
    parts_per_dim,
    # i_snapshots=[-2,-1],
    # i_snapshots=range(1, 33+4, 8),
    i_snapshots=range(1, 33+4, 4),
    return_hydro=True,
)

cosmo = out_dict["cosmo"]
scales = out_dict["scales"]

dm_poss = out_dict["dm_poss"]
dm_vels = out_dict["dm_vels"]

gas_poss = out_dict["gas_poss"]
gas_vels = out_dict["gas_vels"]

Using snapshots ['/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2/snapshot_018.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2/snapshot_034.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2/snapshot_042.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2/snapshot_050.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2/snapshot_058.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2/snapshot_066.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2/snapshot_074.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2/snapshot_082.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2/snapshot_090.hdf5']
Selecting 262144 dark matter (deterministic)
Selecting 262144 gas particles (random)


loading snapshots: 100%|██████████| 9/9 [01:04<00:00,  7.13s/it]


In [6]:
@nnx.jit(static_argnames=("loss_fn",))
def train_step(model, optimizer, loss_fn):
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)

    return loss

losses = []

In [7]:
from jaxpm.graph import get_edges

# edges = get_edges(gas_poss, k=8)
# edges = (edges[0][-1], edges[1][-1], edges[2][-1])

edges = get_edges(gas_poss, scales, k=4)

In [8]:
def solve_ode_diffrax(model, architecture):
# def solve_ode_diffrax(model):
    res = diffeqsolve(
            # terms=ODETerm(hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, integrator_type="diffrax", architecture=architecture)),
            terms=ODETerm(hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, integrator_type="diffrax", precomputed_edges=edges, architecture=architecture)),
            # terms=ODETerm(hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, integrator_type="diffrax", force_type="table")),
            # terms=ODETerm(hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, edges=edges, integrator_type="diffrax", force_type="gnn")),
            solver=LeapfrogMidpoint(),
            t0=scales[0],
            t1=scales[-1],
            dt0=0.01,
            # dt0=0.005,
            # dt0=0.1,
            y0=jnp.stack([dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], axis=0),
            saveat=SaveAt(ts=scales),
            # args={"gas_latent": np.random.random(parts_per_dim**3)},
            # args={"gas_latent": jnp.ones(parts_per_dim**3)},
            # args={"gas_latent": None},
            args={},
            # adjoint=diffrax.RecursiveCheckpointAdjoint(checkpoints),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys
    res = jnp.transpose(res, (1,0,2,3))
    
    return res

def solve_ode_jax(model):
    res = odeint(
        hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, force_type="table"), 
        [dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]],
        scales,
        rtol=1e-2, 
        atol=1e-2
    )

    return res

In [9]:
vcic_paint = jax.vmap(cic_paint, in_axes=(None,0,None))
vcic_read = jax.vmap(cic_read, in_axes=(0,0))

# loss

### CAMELS ground truth

In [10]:
# per-particle reference
ref_pos = jnp.stack(gas_poss, axis=0)
ref_vel = jnp.stack(gas_vels, axis=0)

# field-level reference
ref_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, cosmo.Omega_b / cosmo.Omega_c)

# power spectrum reference
vpower_spectrum = jax.vmap(
    lambda fields: 
        power_spectrum(
            compensate_cic(fields),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
_, ref_cls = vpower_spectrum(ref_rho)

### particle-level

In [11]:
def particle_loss_fn(model, architecture):
# def particle_loss_fn(model):
    res = solve_ode_diffrax(model, architecture)
    # res = solve_ode_diffrax(model)

    # pos_loss = jnp.sum((res[2] - ref_pos)**2, axis=-1)
    pos_loss = jnp.sum((res[2]%64 - ref_pos)**2, axis=-1)

    pos_loss = jnp.where(pos_loss < mesh_per_dim//2, pos_loss, 0.)
    # pos_loss *= jnp.expand_dims(scales, axis=1)
    pos_loss = jnp.mean(pos_loss)

    vel_loss = jnp.sum((res[3] - ref_vel)**2, axis=-1)
    vel_loss = jnp.where(vel_loss < mesh_per_dim//2, vel_loss, 0.)
    # vel_loss *= jnp.expand_dims(scales, axis=1)
    vel_loss = jnp.mean(vel_loss)

    res_rho = vcic_paint(jnp.zeros(mesh_shape), res[2], cosmo.Omega_b / cosmo.Omega_c)
    _, res_cls = vpower_spectrum(res_rho)
    cl_loss = jnp.mean(jnp.sum((res_cls/ref_cls - 1)**2, axis=-1))
    
    return pos_loss + 0.01 * vel_loss + 0.1 * cl_loss
    # return pos_loss + 0.01 * vel_loss + 0.01 * cl_loss
    # return cl_loss
    # return pos_loss + 0.01 * vel_loss
    # return pos_loss + 0.01 * vel_loss
    # return pos_loss + vel_loss
    # return pos_loss


### field-level

In [12]:
# def field_loss_fn(model):
#     res = solve_ode_diffrax(model)

#     rho = vcic_paint(jnp.zeros(mesh_shape), res[2], cosmo.Omega_b / cosmo.Omega_c)
#     rho_loss = jnp.nanmean((rho - ref_rho)**2)
    
#     # eps = 1e-5
#     # rho_loss = jnp.mean((jnp.log1p(rho + eps) - jnp.log1p(ref_rho + eps))**2)
    
#     # _, res_cls = vpower_spectrum(res_rho)
#     # cl_loss = jnp.mean(jnp.sum((res_cls/ref_cls - 1)**2, axis=-1))
    
#     return rho_loss
#     # return rho_loss + 0.1 * cl_loss


# architecture

### MLP

In [13]:
with_latent = False

model = MLP(
    d_in=3 + with_latent, 
    d_out=1 + with_latent, 
    d_hidden=64, 
    n_hidden=4, 
    rngs=nnx.Rngs(0)
)
architecture = "mlp"

### MLP + CNN

In [14]:
# mlp = MLP(
#     d_in=3,
#     d_out=8, 
#     d_hidden=64, 
#     n_hidden=4, 
#     rngs=nnx.Rngs(0)
# )

# cnn = CNN(
#     d_in=2,
#     d_out=8,
#     d_hidden=8,
#     num_layers=0,
#     kernel_size=(4, 4, 4),
#     strides=1,
#     rngs=nnx.Rngs(0)
# )

# model = HybridNet(
#     mlp,
#     cnn,
#     d_out=1,
#     rngs=nnx.Rngs(0)
# )

# architecture = "mlp+cnn"

### GNN

on the fly

In [15]:
# model = GATGNN(
#     d_node=3,
#     d_edge=1,
#     d_query=8,
#     n_hidden=4,
#     d_out=1,
#     rngs=nnx.Rngs(0),
# )

# architecture = "gnn"

# training

In [16]:
total_steps = 200
# learning_rate = 1e-3
learning_rate = optax.cosine_decay_schedule(
    init_value=1e-3, 
    decay_steps=total_steps, 
    alpha=0.1
)
clip_norm = 1

optimizer = nnx.Optimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(clip_norm),
        optax.adam(learning_rate)
    )
)

losses = []
loss_fn = lambda model: particle_loss_fn(model, architecture)

In [ ]:
for i in (pbar := tqdm.tqdm(range(total_steps))):  
    loss = train_step(model, optimizer, loss_fn)

    losses.append(loss)
    pbar.set_description(f"Loss: {loss:.4f}")

fig, ax = plt.subplots()
ax.plot(losses)
ax.set(yscale="log")

  0%|          | 0/200 [00:00<?, ?it/s]

No latent variable
No latent variable
No latent variable


2025-01-30 20:09:34.732529: E external/xla/xla/service/slow_operation_alarm.cc:65] Constant folding an instruction is taking > 1s:

  %negate.56 = f32[9,262144,3]{2,1,0} negate(f32[9,262144,3]{2,1,0} %constant.163)

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
2025-01-30 20:09:40.053878: E external/xla/xla/service/slow_operation_alarm.cc:133] The operation took 6.321433946s
Constant folding an instruction is taking > 1s:

  %negate.56 = f32[9,262144,3]{2,1,0} negate(f32[9,262144,3]{2,1,0} %constant.163)

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some gu

### checkpointing

In [ ]:
# # see https://flax.readthedocs.io/en/latest/guides/checkpointing.html
# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/hpm_sim_cv0.jx")
# checkpointer = ocp.StandardCheckpointer()

In [ ]:
# _, params = nnx.split(model)
# checkpointer.save(checkpoint_file, params, force=True)

In [ ]:
# abstract_model = nnx.eval_shape(lambda: model)
# graphdef, abstract_params = nnx.split(abstract_model)

# params = checkpointer.restore(checkpoint_file, abstract_params)
# model = nnx.merge(graphdef, params)

# run the simulation

In [ ]:
pm_ode = hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, "odeint", gravity_only=True)

# res = odeint(pm_ode, [dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], scales, {"gas_latent": jnp.ones(parts_per_dim**3)}, rtol=1e-5, atol=1e-5)
res = odeint(pm_ode, [dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], scales, {}, rtol=1e-5, atol=1e-5)
pm_dm_poss, pm_dm_vels, pm_gas_poss, pm_gas_vels = res[0], res[1], res[2], res[3]

In [ ]:
# hpm_ode = hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, "odeint", architecture=architecture)
# # hpm_ode = hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, "odeint", force_type="gnn", edges=edges)
# # hpm_ode = lambda state, scale: gnn_hpm_ode(scale, state, None)

# res = odeint(hpm_ode, [dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], scales, {"gas_latent": jnp.ones(parts_per_dim**3)}, rtol=1e-5, atol=1e-5, mxstep=20)
# # res = odeint(hpm_ode, [dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], scales, {}, rtol=1e-5, atol=1e-5, mxstep=20)
# hpm_dm_poss, hpm_dm_vels, hpm_gas_poss, hpm_gas_vels = res[0], res[1], res[2], res[3]

In [ ]:
res = diffeqsolve(
        terms=ODETerm(hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, integrator_type="diffrax", precomputed_edges=edges, architecture=architecture)),
        solver=LeapfrogMidpoint(),
        t0=scales[0],
        t1=scales[-1],
        dt0=0.01,
        y0=jnp.stack([dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], axis=0),
        saveat=SaveAt(ts=scales),
        args={"gas_latent": jnp.ones(parts_per_dim**3)},
        max_steps=100,
        stepsize_controller=ConstantStepSize(),
    )
res = res.ys
res = jnp.transpose(res, (1,0,2,3))

hpm_dm_poss, hpm_dm_vels, hpm_gas_poss, hpm_gas_vels = res[0], res[1], res[2], res[3]

In [ ]:
plotting.compare_particle_evolution(
    mesh_shape, 
    scales, 
    jnp.stack([gas_poss, pm_gas_poss, hpm_gas_poss], axis=0), 
    title="gas",
    col_titles=["CAMELS", "gravity", "gravity + pressure"],
    include_pk=True,
    include_reference=True,
)

In [ ]:
# k, cross_i = cross_correlation_coefficients(
#       (cic_paint(jnp.zeros(mesh_shape), gas_poss[-1])),
#       (cic_paint(jnp.zeros(mesh_shape), pm_gas_poss[-1])),
#       boxsize=np.array([25.] * 3),
#       kmin=np.pi / 25.,
#       dk=2 * np.pi / 25.
# )

In [ ]:
# plotting.compare_particle_evolution(
#     mesh_shape, 
#     scales, 
#     jnp.stack([gas_poss, hpm_gas_poss], axis=0), 
#     title="gas",
#     col_titles=["CAMELS", "gravity + pressure"],
#     include_pk=True,
# )

In [ ]:
# plotting.compare_particle_evolution(
#     mesh_shape, 
#     scales, 
#     jnp.stack([gas_poss, pm_gas_poss], axis=0), 
#     title="gas",
#     col_titles=["CAMELS", "gravity"],
#     include_pk=True,
# )

# trash

constant graph for every training step

In [ ]:
from jaxpm.hpm import hpm_gnn_forces

In [ ]:
from jaxpm.graph import get_edges, get_graph_given_edges

def get_graph(scale, pos, rho, fscalar, k=4, boxsize=None):
    scale = jax.lax.stop_gradient(scale)
    pos = jax.lax.stop_gradient(pos)
    rho = jax.lax.stop_gradient(rho)
    fscalar = jax.lax.stop_gradient(fscalar)

    edges = get_edges(pos, scale, k, boxsize=boxsize)
    graph = get_graph_given_edges(scale, edges, rho, fscalar)

    return graph


In [ ]:
def get_graph(scale, pos, rho, fscalar, k=4, boxsize=None):
    scale = jax.lax.stop_gradient(scale)
    pos = jax.lax.stop_gradient(pos)
    rho = jax.lax.stop_gradient(rho)
    fscalar = jax.lax.stop_gradient(fscalar)

    print(scale.shape)
    print(pos.shape)
    print(rho.shape)
    print(fscalar.shape)

    n_node = pos.shape[0]
    n_edge = k * n_node

    k_dist, k_idx = jax_get_knn(pos, k, boxsize=boxsize)

    node_features = jnp.stack([jnp.tile(scale, n_node), jnp.log10(rho), jnp.arcsinh(fscalar / 100)], axis=-1)
    edge_features = k_dist.reshape(-1, 1)

    senders = k_idx.reshape(-1)
    receivers = jnp.repeat(jnp.arange(n_node, dtype=jnp.int32), k)

    graph = jraph.GraphsTuple(
        nodes=node_features,
        edges=edge_features,
        senders=senders,
        receivers=receivers,
        n_node=n_node,
        n_edge=n_edge,
        globals=None,
    )

    return graph


In [ ]:
# def local_hpm_ode(scale, state, edges):
# edges = (get_connectivities(gas_poss, k=4)[0][-1], get_connectivities(gas_poss, k=4)[1][-1], get_connectivities(gas_poss, k=4)[2][-1])
edges = get_connectivities(gas_poss, k=4)
edges = (edges[0][-1], edges[1][-1], edges[2][-1])

def gnn_hpm_ode(scale, state, args):
    dm_pos, dm_vel, gas_pos, gas_vel = state
    
    dm_force, gas_force = hpm_gnn_forces(
        scale, dm_pos, gas_pos, mesh_shape, cosmo, model#, edges=edges
    )

    dm_force *= 1.5 * cosmo.Omega_m
    gas_force *= 1.5 * cosmo.Omega_m

    # update the positions (drift)
    pos_fac = 1.0 / (scale**3 * jnp.sqrt(jc.background.Esqr(cosmo, scale)))
    d_dm_pos = pos_fac * dm_vel
    d_gas_pos = pos_fac * gas_vel

    # update the velocities (kick)
    vel_fac = 1.0 / (scale**2 * jnp.sqrt(jc.background.Esqr(cosmo, scale)))
    d_dm_vel = vel_fac * dm_force
    d_gas_vel = vel_fac * gas_force

    return jnp.stack([d_dm_pos, d_dm_vel, d_gas_pos, d_gas_vel])


In [ ]:
model = AttentionGNN(
    d_node=2,
    d_edge=1,
    d_query=16,
    n_hidden=1,
    d_out=1,
    rngs=nnx.Rngs(0),
)

In [ ]:
i = -1
def loss_on_the_fly(model):
    dm_force, gas_force = hpm_gnn_forces(scales[i], dm_poss[i], gas_poss[i], mesh_shape, cosmo, model, edges=None)
    return jnp.mean(dm_force**2) + jnp.mean(gas_force**2)

In [ ]:
# nnx.grad(loss_on_the_fly)(model)

In [ ]:
i = -1
def loss_prebuilt(model):
    dm_force, gas_force = hpm_gnn_forces(scales[i], dm_poss[i], gas_poss[i], mesh_shape, cosmo, model, edges=edges)
    return jnp.mean(dm_force**2) + jnp.mean(gas_force**2)

In [ ]:
# nnx.grad(loss_prebuilt)(model)

In [ ]:
total_steps = 10
learning_rate = 1e-3
# learning_rate = optax.cosine_decay_schedule(
#     init_value=1e-3, 
#     decay_steps=total_steps, 
#     alpha=0.1
# )
clip_norm = 1

optimizer = nnx.Optimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(clip_norm),
        optax.adam(learning_rate)
    )
)

losses = []

In [ ]:
def gnn_particle_loss_fn(model):
    res = diffeqsolve(
            terms=ODETerm(gnn_hpm_ode),
            solver=LeapfrogMidpoint(),
            t0=scales[0],
            t1=scales[-1],
            dt0=0.01,
            y0=jnp.stack([dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], axis=0),
            saveat=SaveAt(ts=scales),
            # adjoint=diffrax.RecursiveCheckpointAdjoint(checkpoints),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys
    res = jnp.transpose(res, (1,0,2,3))

    pos_loss = jnp.sum((res[2]%64 - ref_pos)**2, axis=-1)

    pos_loss = jnp.where(pos_loss < mesh_per_dim//2, pos_loss, 0.)
    pos_loss = jnp.mean(pos_loss)

    vel_loss = jnp.sum((res[3] - ref_vel)**2, axis=-1)
    vel_loss = jnp.where(vel_loss < mesh_per_dim//2, vel_loss, 0.)
    vel_loss = jnp.mean(vel_loss)

    res_rho = vcic_paint(jnp.zeros(mesh_shape), res[2], cosmo.Omega_b / cosmo.Omega_c)
    _, res_cls = vpower_spectrum(res_rho)
    cl_loss = jnp.mean(jnp.sum((res_cls/ref_cls - 1)**2, axis=-1))
    
    # return pos_loss + 0.01 * vel_loss + 0.1 * cl_loss
    return pos_loss

loss = []

In [ ]:
for i in (pbar := tqdm.tqdm(range(total_steps))):
    loss = train_step(model, optimizer, gnn_particle_loss_fn)

    losses.append(loss)
    pbar.set_description(f"Loss: {loss:.4f}")

    # print(grads)

fig, ax = plt.subplots()
ax.plot(losses)
ax.set(yscale="log")

In [ ]:
# def get_per_particle_inputs(dm_poss, gas_poss):
#     # rho (exactly like in PM)
#     rho_dm = vcic_paint(jnp.zeros(mesh_shape), dm_poss, None)
#     rho_gas = vcic_paint(jnp.zeros(mesh_shape), gas_poss, cosmo.Omega_b / cosmo.Omega_c)
#     rho = rho_dm + rho_gas
#     gas_rho = vcic_read(rho, gas_poss)
    
#     # fscalar
#     kvec = fftk(mesh_shape)
#     delta_k = jax.vmap(jnp.fft.rfftn, in_axes=0)(rho)
#     fscalar = jax.vmap(jnp.fft.irfftn, in_axes=0)(delta_k * invnabla_kernel(kvec))
#     gas_fscalar = vcic_read(fscalar, gas_poss)

#     return gas_rho, gas_fscalar

# def get_graphs(dm_poss, gas_poss, k=16):
#     n_node = gas_poss.shape[1]
#     n_edge = k * n_node
    
#     k_dist, k_idx = jax.vmap(knn_scipy, in_axes=(0, None))(gas_poss, k)

#     gas_rho, gas_fscalar = get_per_particle_inputs(dm_poss, gas_poss)
#     node_features = jnp.stack([jnp.log10(gas_rho), jnp.arcsinh(gas_fscalar/100)], axis=-1)
#     edge_features = k_dist.reshape(-1, 1)
    
#     senders = k_idx.reshape(-1)
#     receivers = jnp.repeat(jnp.arange(n_node, dtype=jnp.int32), k)
    
#     graphs = jraph.GraphsTuple(
#         nodes=node_features,
#         edges=edge_features,
#         senders=senders,
#         receivers=receivers,
#         n_node=n_node,
#         n_edge=n_edge,
#         globals=None,
#     )

#     return graph

# def get_graphs(edges, rho, fscalar, k=16):
#     n_node = gas_poss.shape[1]
#     n_edge = k * n_node
    
#     k_dist, k_idx = jax.vmap(knn_scipy, in_axes=(0, None))(gas_poss, k)

#     node_features = jnp.stack([jnp.log10(gas_rho), jnp.arcsinh(gas_fscalar/100)], axis=-1)
#     edge_features = k_dist.reshape(-1, 1)
    
#     senders = k_idx.reshape(-1)
#     receivers = jnp.repeat(jnp.arange(n_node, dtype=jnp.int32), k)
    
#     graphs = jraph.GraphsTuple(
#         nodes=node_features,
#         edges=edge_features,
#         senders=senders,
#         receivers=receivers,
#         n_node=n_node,
#         n_edge=n_edge,
#         globals=None,
#     )

#     return graph

# def get_graph(edges, rho, fscalar, k=4):
#     n_node = rho.shape[0]
#     n_edge = edges[0].shape[0]

#     node_features = jnp.stack([jnp.log10(rho), jnp.arcsinh(fscalar/100)], axis=-1)

#     # print(node_features.shape)
        
#     graph = jraph.GraphsTuple(
#         nodes=node_features,
#         edges=edges[0],
#         senders=edges[1],
#         receivers=edges[2],
#         n_node=n_node,
#         n_edge=n_edge,
#         globals=None,
#     )

#     return graph

# def get_connectivities(poss, k=16):
    # k_dist, k_idx = jax.vmap(knn_scipy, in_axes=(0, None))(poss, k)

    # edge_features = k_dist.reshape(poss.shape[0], -1, 1)
    # senders = k_idx.reshape(poss.shape[0], -1)
    # receivers = jnp.repeat(jnp.arange(n_node, dtype=jnp.int32), k)

    # return edge_features, senders, receivers

# def get_connectivities(poss, k=4):
#     n_node = poss.shape[1]
    
#     # TODO vectorize this properly
#     edge_features, senders, receivers = [], [], []
#     for i in range(poss.shape[0]):
#         k_dist, k_idx = jax_get_knn(poss[i], k)
#         edge_feature = k_dist.reshape(-1, 1)
#         sender = k_idx.reshape(-1)
#         receiver = jnp.repeat(jnp.arange(n_node, dtype=jnp.int32), k)

#         edge_features.append(edge_feature)
#         senders.append(sender)
#         receivers.append(receiver)

#     edge_features = jnp.stack(edge_features, axis=0)
#     senders = jnp.stack(senders, axis=0)
#     receivers = jnp.stack(receivers, axis=0)

#     return edge_features, senders, receivers

In [ ]:
get_connectivities(gas_poss, k=4)[0].shape

In [ ]:
def gnn_hpm_forces(scale, dm_pos, gas_pos, mesh_shape, cosmo, model, edges, r_split=0):
    kvec = fftk(mesh_shape)

    rho_dm = cic_paint(jnp.zeros(mesh_shape), dm_pos)
    rho_gas = cic_paint(jnp.zeros(mesh_shape), gas_pos, weight=cosmo.Omega_b / cosmo.Omega_c)
    rho_tot = rho_dm + rho_gas

    # gravitational potential
    rho_k_tot = jnp.fft.rfftn(rho_tot)
    phi_k = rho_k_tot * invlaplace_kernel(kvec) * longrange_kernel(kvec, r_split=r_split)

    def gravity(pos):
        return jnp.stack(
            [cic_read(jnp.fft.irfftn(gradient_kernel(kvec, i) * phi_k), pos) for i in range(len(kvec))],
            axis=-1,
        )

    dm_force = -gravity(dm_pos)
    gas_force = -gravity(gas_pos)

    # pressure force
    gas_rho_tot = cic_read(rho_tot, gas_pos)
    gas_fscalar = cic_read(jnp.fft.irfftn(rho_k_tot * invnabla_kernel(kvec)), gas_pos)
    
    gas_inputs = get_graph(edges, gas_rho_tot, gas_fscalar)
    
    gas_P = 10 ** jnp.squeeze(model(gas_inputs).nodes)

    # print(gas_inputs)
    # print(gas_P)
    # print(model(gas_inputs).nodes)

    gas_rho = cic_read(rho_gas, gas_pos)
    P_k = jnp.fft.rfftn(cic_paint(jnp.zeros(mesh_shape), gas_pos, weight=gas_P / gas_rho))

    def pressure(pos):
        nabla_P = jnp.stack(
            [cic_read(jnp.fft.irfftn(gradient_kernel(kvec, i) * P_k), pos) for i in range(len(kvec))],
            axis=-1,
        )
        return nabla_P / jnp.expand_dims(gas_rho, axis=-1)

    gas_force -= pressure(gas_pos)

    return dm_force, gas_force


In [ ]:
# gnn_hpm_forces(scales[-3], dm_poss[-3], gas_poss[-3], mesh_shape, cosmo, model, edges)

In [ ]:
# get_connectivities(gas_poss, k=4)[0][-1]

In [ ]:
# def local_hpm_ode(scale, state, edges):
# edges = (get_connectivities(gas_poss, k=4)[0][-1], get_connectivities(gas_poss, k=4)[1][-1], get_connectivities(gas_poss, k=4)[2][-1])
edges = get_connectivities(gas_poss, k=4)
edges = (edges[0][-1], edges[1][-1], edges[2][-1])

def gnn_hpm_ode(scale, state, args):
    dm_pos, dm_vel, gas_pos, gas_vel = state
    
    dm_force, gas_force = gnn_hpm_forces(
        scale, dm_pos, gas_pos, mesh_shape, cosmo, model, edges
    )

    dm_force *= 1.5 * cosmo.Omega_m
    gas_force *= 1.5 * cosmo.Omega_m

    # update the positions (drift)
    pos_fac = 1.0 / (scale**3 * jnp.sqrt(jc.background.Esqr(cosmo, scale)))
    d_dm_pos = pos_fac * dm_vel
    d_gas_pos = pos_fac * gas_vel

    # update the velocities (kick)
    vel_fac = 1.0 / (scale**2 * jnp.sqrt(jc.background.Esqr(cosmo, scale)))
    d_dm_vel = vel_fac * dm_force
    d_gas_vel = vel_fac * gas_force

    return jnp.stack([d_dm_pos, d_dm_vel, d_gas_pos, d_gas_vel])


In [ ]:
model = AttentionGNN(
    d_node=2,
    d_edge=1,
    d_query=16,
    n_hidden=1,
    d_out=1,
    rngs=nnx.Rngs(0),
)

In [ ]:
total_steps = 100
learning_rate = 1e-3
# learning_rate = optax.cosine_decay_schedule(
#     init_value=1e-3, 
#     decay_steps=total_steps, 
#     alpha=0.1
# )
clip_norm = 1

optimizer = nnx.Optimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(clip_norm),
        optax.adam(learning_rate)
    )
)

losses = []

In [ ]:
def gnn_particle_loss_fn(model):
    res = diffeqsolve(
            terms=ODETerm(gnn_hpm_ode),
            solver=LeapfrogMidpoint(),
            t0=scales[0],
            t1=scales[-1],
            dt0=0.01,
            y0=jnp.stack([dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], axis=0),
            saveat=SaveAt(ts=scales),
            # adjoint=diffrax.RecursiveCheckpointAdjoint(checkpoints),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys
    res = jnp.transpose(res, (1,0,2,3))

    pos_loss = jnp.sum((res[2]%64 - ref_pos)**2, axis=-1)

    pos_loss = jnp.where(pos_loss < mesh_per_dim//2, pos_loss, 0.)
    pos_loss = jnp.mean(pos_loss)

    vel_loss = jnp.sum((res[3] - ref_vel)**2, axis=-1)
    vel_loss = jnp.where(vel_loss < mesh_per_dim//2, vel_loss, 0.)
    vel_loss = jnp.mean(vel_loss)

    res_rho = vcic_paint(jnp.zeros(mesh_shape), res[2], cosmo.Omega_b / cosmo.Omega_c)
    _, res_cls = vpower_spectrum(res_rho)
    cl_loss = jnp.mean(jnp.sum((res_cls/ref_cls - 1)**2, axis=-1))
    
    # return pos_loss + 0.01 * vel_loss + 0.1 * cl_loss
    return pos_loss

loss = []

In [ ]:
for i in (pbar := tqdm.tqdm(range(total_steps))):
    loss = train_step(model, optimizer, gnn_particle_loss_fn)

    losses.append(loss)
    pbar.set_description(f"Loss: {loss:.4f}")

    # print(grads)

fig, ax = plt.subplots()
ax.plot(losses)
ax.set(yscale="log")

In [ ]:
# def scipy_get_k_nearest_neighbor_idx(points, k, distance_upper_bound=np.inf, boxsize=None, workers=-1, leafsize=10):
#     kd_tree = scipy.spatial.cKDTree(data=points, boxsize=boxsize, leafsize=leafsize)
#     distances, idx = kd_tree.query(x=points, k=int(k), workers=workers, distance_upper_bound=distance_upper_bound)
#     return distances.astype(points.dtype), idx.astype(np.int32)

# @partial(jax.jit, static_argnames=["k", "distance_upper_bound", "boxsize", "workers", "leafsize"])
# def jax_callback_get_k_nearest_neighbor_idx(
#     points, k, distance_upper_bound=np.inf, boxsize=None, workers=-1, leafsize=10
# ):
#     shape = (jnp.shape(points)[0], k)
#     distance_type = jax.ShapeDtypeStruct(shape, points.dtype)
#     idx_type = jax.ShapeDtypeStruct(shape, jnp.int32)
#     return jax.pure_callback(
#         scipy_get_k_nearest_neighbor_idx,
#         (distance_type, idx_type),
#         points,
#         k,
#         distance_upper_bound,
#         boxsize,
#         workers,
#         leafsize,
#     )

# knn_scipy = jax_callback_get_k_nearest_neighbor_idx

In [ ]:
# def get_graph(pos, rho, fscalar, k=16):
#     n_node = pos.shape[0]
#     n_edge = k * n_node
    
#     k_dist, k_idx = knn_scipy(pos, k)
#     # k_dist, k_idx = knn_scipy(pos, k, boxsize=mesh_per_dim)

#     node_features = jnp.stack([jnp.log10(rho), jnp.arcsinh(fscalar/100)], axis=-1)
#     edge_features = k_dist.reshape(-1, 1)
    
#     senders = k_idx.reshape(-1)
#     receivers = jnp.repeat(jnp.arange(n_node, dtype=jnp.int32), k)
    
#     graph = jraph.GraphsTuple(
#         nodes=node_features,
#         edges=edge_features,
#         senders=senders,
#         receivers=receivers,
#         n_node=n_node,
#         n_edge=n_edge,
#         globals=None,
#     )

#     return graph

# def hpm_gnn_forces(scale, dm_pos, gas_pos, mesh_shape, cosmo, model, gravity_only=False, r_split=0):
#     kvec = fftk(mesh_shape)

#     rho_dm = cic_paint(jnp.zeros(mesh_shape), dm_pos)
#     rho_gas = cic_paint(jnp.zeros(mesh_shape), gas_pos, weight=cosmo.Omega_b / cosmo.Omega_c)
#     rho_tot = rho_dm + rho_gas

#     # gravitational potential
#     rho_k_tot = jnp.fft.rfftn(rho_tot)
#     phi_k = rho_k_tot * invlaplace_kernel(kvec) * longrange_kernel(kvec, r_split=r_split)

#     def gravity(pos):
#         return jnp.stack(
#             [cic_read(jnp.fft.irfftn(gradient_kernel(kvec, i) * phi_k), pos) for i in range(len(kvec))],
#             axis=-1,
#         )

#     dm_force = -gravity(dm_pos)
#     gas_force = -gravity(gas_pos)

#     # pressure force
#     if not gravity_only:
#         gas_rho_tot = cic_read(rho_tot, gas_pos)
#         gas_fscalar = cic_read(jnp.fft.irfftn(rho_k_tot * invnabla_kernel(kvec)), gas_pos)
#         gas_inputs = get_graph(gas_pos, gas_rho_tot, gas_fscalar)
        
#         gas_P = 10 ** jnp.squeeze(model(gas_inputs))

#         gas_rho = cic_read(rho_gas, gas_pos)
#         P_k = jnp.fft.rfftn(cic_paint(jnp.zeros(mesh_shape), gas_pos, weight=gas_P / gas_rho))

#         def pressure(pos):
#             nabla_P = jnp.stack(
#                 [cic_read(jnp.fft.irfftn(gradient_kernel(kvec, i) * P_k), pos) for i in range(len(kvec))],
#                 axis=-1,
#             )
#             return nabla_P / jnp.expand_dims(gas_rho, axis=-1)

#         gas_force -= pressure(gas_pos)

#     return dm_force, gas_force


In [ ]:
# def get_per_particle_inputs(dm_pos, gas_pos):
#     # rho (exactly like in PM)
#     rho_dm = cic_paint(jnp.zeros(mesh_shape), dm_pos, 1)
#     rho_gas = cic_paint(jnp.zeros(mesh_shape), gas_pos, cosmo.Omega_b / cosmo.Omega_c)
#     rho = rho_dm + rho_gas
#     gas_rho = cic_read(rho, gas_pos)
    
#     # fscalar
#     kvec = fftk(mesh_shape)
#     delta_k = jnp.fft.rfftn(rho)
#     fscalar = jnp.fft.irfftn(delta_k * invnabla_kernel(kvec))
#     gas_fscalar = cic_read(fscalar, gas_pos)

#     return gas_rho, gas_fscalar

# def get_graph(dm_pos, gas_pos, k=16):
#     n_node = gas_poss.shape[0]
#     n_edge = k * n_node
    
#     k_dist, k_idx = knn_scipy(gas_pos, k)
#     # k_dist, k_idx = knn_scipy(gas_pos, k, boxsize=mesh_per_dim)

#     gas_rho, gas_fscalar = get_per_particle_inputs(dm_pos, gas_pos)
#     node_features = jnp.stack([jnp.log10(gas_rho), jnp.arcsinh(gas_fscalar/100)], axis=-1)
#     edge_features = k_dist.reshape(-1, 1)
    
#     senders = k_idx.reshape(-1)
#     receivers = jnp.repeat(jnp.arange(n_node, dtype=jnp.int32), k)
    
#     graph = jraph.GraphsTuple(
#         nodes=node_features,
#         edges=edge_features,
#         senders=senders,
#         receivers=receivers,
#         n_node=n_node,
#         n_edge=n_edge,
#         globals=None,
#     )

#     return graph

In [ ]:
# get_graph(dm_poss[0], gas_poss[0])

In [ ]:
# get_per_particle_inputs(dm_poss[0], gas_poss[0])

In [ ]:
# @jax.custom_vjp
# def jax_get_k_nearest_neighbor(positions, k):
#     def forward(positions, k):
#         # Use scipy or any other library in pure_callback
#         def compute_neighbors(positions_np):
#             from scipy.spatial import cKDTree

#             tree = cKDTree(positions_np)
#             return tree.query(positions_np, k=k + 1)[1][:, 1:]  # Skip self

#         return jax.pure_callback(compute_neighbors, jnp.zeros((positions.shape[0], k), dtype=jnp.int32), positions, k)

#     return forward(positions, k)


# def fwd(positions, k):
#     neighbors = jax_get_k_nearest_neighbor(positions, k)
#     return neighbors, None  # No intermediate state needed


# def bwd(_, g):
#     return g * 0, 0  # Zero gradient (effectively stops gradients here)


# jax_get_k_nearest_neighbor.defvjp(fwd, bwd)

In [ ]:
# k_dist, k_idx = jax_get_k_nearest_neighbor(gas_poss[0], k=4)

### annax

In [ ]:
import numpy as np
import annax

# Generate some random high-dimensional data
data = np.random.random((1000, 128))

# Create an Annax index with the default configuration
index = annax.Index(data)

# Query for the 10 nearest neighbors of a random vector
query = np.random.random(128)
neighbors, distances = index.search(query, k=10)

In [ ]:
from jaxpm.gnn import jax_get_knn
import annax

pos = gas_poss[-1]
k = 16

In [ ]:
dist, idx = jax_get_knn(pos, k)

In [ ]:
dist.shape

In [ ]:
idx.shape

In [ ]:
%%timeit
jax_get_knn(pos, k)

In [ ]:
neighbors

In [ ]:
index = annax.Index(pos)

In [ ]:
pos.shape

In [ ]:
idx, dist = index.search(pos, k=k)

In [ ]:
idx.shape

In [ ]:
dist.shape

In [ ]:
pos[:10].shape

In [ ]:
jax.vmap(index.search, in_axes=(0, None))(pos[:10], k=k)

In [ ]:
neighbors

In [ ]:
pos.shape

# static graph for training

In [ ]:
from jaxpm.gnn import get_edges

In [ ]:
edges = get_edges(gas_poss, scales)

In [ ]:
edges["features"].shape

In [ ]:
edges["scales"].shape

In [ ]:
gas_poss.ndim

In [ ]:
scale = jnp.array([0])

In [ ]:
jnp.expand_dims(scale, 1).shape